# Cervical Cancer Stage Classification on Kaggle

This notebook launches the backend training script with Kaggle-friendly paths. It looks for the repository, finds the dataset, and writes checkpoints to the Kaggle working directory.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

def detect_repo_root() -> Path:
    candidates = [Path('/kaggle/working'), Path('/kaggle/input'), Path.cwd()]
    for base in candidates:
        if not base.exists():
            continue
        if (base / 'backend').exists():
            return base
        for child in base.iterdir():
            if child.is_dir() and (child / 'backend').exists():
                return child
    return Path.cwd()

REPO_ROOT = detect_repo_root()
BACKEND_DIR = REPO_ROOT / 'backend'
os.chdir(REPO_ROOT)
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

print('REPO_ROOT:', REPO_ROOT)
print('BACKEND_DIR:', BACKEND_DIR)
print('Python:', sys.executable)

In [ ]:
CLASS_NAMES = ['Normal', 'CIN1', 'CIN2', 'CIN3', 'Cancer']


def _class_count(base: Path) -> int:
    try:
        return sum(1 for name in CLASS_NAMES if (base / name).is_dir())
    except Exception:
        return 0


def _looks_like_dataset_root(base: Path) -> bool:
    if not base.exists() or not base.is_dir():
        return False

    if _class_count(base) == len(CLASS_NAMES):
        return True

    train_dir = base / 'train'
    val_dir = base / 'val'
    test_dir = base / 'test'
    if train_dir.is_dir() and _class_count(train_dir) == len(CLASS_NAMES):
        return True
    if val_dir.is_dir() and _class_count(val_dir) == len(CLASS_NAMES):
        return True
    if test_dir.is_dir() and _class_count(test_dir) == len(CLASS_NAMES):
        return True
    return False


def find_data_dir() -> Path | None:
    raw_candidates = [
        os.environ.get('DATA_DIR', ''),
        '/kaggle/input/datasets/shubhrawat132/herlevdataset',
        '/kaggle/input/Herlev Dataset',
        '/kaggle/input/herlev-dataset',
        '/kaggle/input/herlevdataset',
        '/kaggle/input/cervical-cancer-stage-classification',
        '/kaggle/input/cervical-cancer-dataset',
        str(REPO_ROOT / 'Herlev Dataset'),
        str(REPO_ROOT / 'data'),
    ]

    best_root: Path | None = None
    best_score = -1

    for candidate_text in raw_candidates:
        if not candidate_text:
            continue
        candidate = Path(candidate_text)
        if not candidate.exists():
            continue

        search_roots = [candidate]
        try:
            search_roots.extend([p for p in candidate.rglob('*') if p.is_dir()])
        except Exception:
            pass

        for root in search_roots:
            if not _looks_like_dataset_root(root):
                continue

            score = 0
            score += _class_count(root) * 100
            score += _class_count(root / 'train') * 10
            score += _class_count(root / 'val') * 5
            score += _class_count(root / 'test') * 5

            if score > best_score:
                best_score = score
                best_root = root / 'train' if (root / 'train').is_dir() and _class_count(root / 'train') == len(CLASS_NAMES) else root

    return best_root


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path('/kaggle/working/Checkpoints') if Path('/kaggle/working').exists() else (REPO_ROOT / 'backend' / 'Checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if DATA_DIR is None:
    raise FileNotFoundError('Could not find the dataset. Set DATA_DIR to your Kaggle input folder and rerun this cell.')

print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Class folders found:', {name: (DATA_DIR / name).exists() for name in CLASS_NAMES})

In [ ]:
train_script = BACKEND_DIR / 'train.py'
command = [
    sys.executable,
    str(train_script),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', '90',
    '--batch-size', '16',
    '--img-size', '224',
    '--backbone', 'tf_efficientnetv2_m',
    '--lr-head', '3e-4',
    '--lr-backbone', '3e-5',
    '--phase1-epochs', '24',
    '--phase2-epochs', '26',
    '--mixup-alpha', '0.30',
    '--focal-gamma', '1.5',
    '--patience', '15',
    '--num-workers', str(min(4, os.cpu_count() or 2)),
]

print('Running:')
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
artifacts = sorted(OUTPUT_DIR.glob('*'))
print('Training artifacts:')
for artifact in artifacts:
    print('-', artifact.name)

metrics_path = OUTPUT_DIR / 'metrics.json'
print('metrics.json exists:', metrics_path.exists())
if metrics_path.exists():
    print('metrics.json saved at', metrics_path)